# 03. Build Trip Analytical Dataset

## Objective

Build the final trip-level analytical dataset by allocating fuel cost to each trip and calculating profitability metrics.

Each row represents one trip and becomes the single source of truth for all downstream business-question notebooks and the Tableau dashboard.

At this stage the dataset adds:

- fuel cost per trip, allocated using FIFO against the company-wide fuel purchase queue
- gross profit and gross margin per trip

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [2]:
# Load environment variables
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True,
    future=True
)

try:
    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))
    print("✅ Successfully connected to MySQL.")
except Exception as e:
    print(f"❌ Connection failed:\n{e}")

✅ Successfully connected to MySQL.


### Load Input Data

`trip_operational_dataset` and `fuel_batches` are both read directly from MySQL — the source of truth for the pipeline. A CSV snapshot of `trip_operational_dataset` is also available at `data/processed/trip_operational_dataset.csv` for reference and reproducibility outside a live database connection.

In [3]:
trip_operational_dataset = pd.read_sql(
    text("SELECT * FROM trip_operational_dataset"),
    engine,
    parse_dates=["date_departure", "date_arrival"]
)

print(f"Dataset successfully loaded.")

trip_operational_dataset.head()

Dataset successfully loaded.


,trip_id,date_departure,date_arrival,status,client_id,client_name,client_region,route_id,origin_city,destination_city,...,truck_capacity_tons,truck_year,driver_id,driver_name,driver_commission_rate,cargo_tons_actual,load_factor_pct,delay_hours,revenue_uah,estimated_fuel_liters
0,TRP-00001,2022-07-01,2022-07-02,completed,CLT-02,ПП ДніпроЗерно,Дніпропетровська,RTE-02,Дніпро,Одеса,...,24.0,2018,DRV-04,Андрій Савченко,0.13,20.76,86.5,1.8,18715.5552,239.59
1,TRP-00002,2022-07-01,2022-07-01,completed,CLT-09,ФГ Золоте Колосся,Хмельницька,RTE-09,Хмельницький,Тернопіль,...,24.0,2018,DRV-04,Андрій Савченко,0.13,16.44,68.5,1.5,3829.8624,56.85
2,TRP-00003,2022-07-01,2022-07-01,completed,CLT-09,ФГ Золоте Колосся,Хмельницька,RTE-09,Хмельницький,Тернопіль,...,24.5,2018,DRV-05,Ігор Шевченко,0.13,16.74,68.3,1.1,3899.7504,54.93
3,TRP-00004,2022-07-01,2022-07-01,delayed,CLT-08,ТОВ Пшеничний Шлях,Чернігівська,RTE-08,Чернігів,Київ,...,24.5,2018,DRV-11,Тарас Гончар,0.13,24.47,99.9,2.1,6526.3937,73.07
4,TRP-00005,2022-07-01,2022-07-01,completed,CLT-07,ФГ Жнива-Поділля,Вінницька,RTE-07,Вінниця,Жмеринка,...,24.5,2018,DRV-05,Ігор Шевченко,0.13,19.47,79.5,0.4,1883.1384,25.50


In [4]:
fuel_batches = pd.read_sql(
    text("SELECT * FROM fuel_batches"),
    engine,
    parse_dates=["purchase_date"]
)

print(f"Fuel batches: {len(fuel_batches):,}")

fuel_batches.head()

Fuel batches: 16


,batch_id,purchase_date,supplier,liters_purchased,price_per_liter_uah,total_cost_uah
0,FB-001,2022-07-01,ОККО Пальне,28826.0,54.2,1562369.2
1,FB-002,2022-08-18,ОККО Пальне,29708.0,52.7,1565611.6
2,FB-003,2022-10-07,ОККО Пальне,29131.0,48.6,1415766.6
3,FB-004,2022-12-05,ОККО Пальне,27969.0,46.4,1297761.6
4,FB-005,2023-02-03,ОККО Пальне,26698.0,47.1,1257475.8


### Quick Quality Check — Fuel Balance

Before running FIFO allocation across the full dataset, confirm that total purchased fuel covers total estimated consumption. If it doesn't, `allocate_fifo_fuel_cost()` will fail partway through with a clearer error — better to catch the imbalance here first.

In [5]:
total_needed = trip_operational_dataset["estimated_fuel_liters"].sum()
total_available = fuel_batches["liters_purchased"].sum()

print(f"Needed:    {total_needed:,.0f} L")
print(f"Available: {total_available:,.0f} L")
print(f"Surplus:   {total_available - total_needed:,.0f} L")

assert total_available >= total_needed, "Not enough purchased fuel to cover estimated consumption."

Needed:    461,447 L
Available: 478,153 L
Surplus:   16,706 L


## FIFO Fuel Cost Allocation

Fuel cost is allocated using `allocate_fifo_fuel_cost()`, imported from `python/fuel_cost_allocation.py`. The function is unit-tested separately (see `python/test_fuel_cost_allocation.py`) and treated here as a trusted building block — this notebook applies it and validates the output, it does not re-implement or re-verify the algorithm itself.

In [6]:
sys.path.append(str(Path("../python").resolve()))
from fuel_cost_allocation import allocate_fifo_fuel_cost

In [7]:
fuel_allocation = allocate_fifo_fuel_cost(
    trips_df=trip_operational_dataset[["trip_id", "date_departure", "estimated_fuel_liters"]],
    fuel_batches_df=fuel_batches
)

fuel_allocation.head()

,trip_id,fuel_cost_uah,fuel_liters_allocated,batches_used
0,TRP-00001,12985.78,239.59,FB-001
1,TRP-00002,3081.27,56.85,FB-001
2,TRP-00003,2977.21,54.93,FB-001
3,TRP-00004,3960.39,73.07,FB-001
4,TRP-00005,1382.10,25.50,FB-001


### QA — Fuel Liters Allocation Check

`fuel_liters_allocated` should match `estimated_fuel_liters` for every trip that received an allocation. A mismatch would mean the FIFO function lost or double-counted liters somewhere.

In [8]:
check = trip_operational_dataset[["trip_id", "estimated_fuel_liters"]].merge(
    fuel_allocation[["trip_id", "fuel_liters_allocated"]],
    on="trip_id",
    how="inner"
)

mismatch = check.loc[
    check["estimated_fuel_liters"].notna()
    & check["fuel_liters_allocated"].notna()
    & ~check["estimated_fuel_liters"].round(2).eq(check["fuel_liters_allocated"].round(2))
]

print(f"Mismatched rows: {len(mismatch)}")
mismatch.head()

Mismatched rows: 0


,trip_id,estimated_fuel_liters,fuel_liters_allocated


## Build Analytical Dataset

Merge fuel cost back onto the operational dataset and calculate trip-level profitability.

In [9]:
trip_analytical_dataset = trip_operational_dataset.merge(
    fuel_allocation[["trip_id", "fuel_cost_uah", "batches_used"]],
    on="trip_id",
    how="left"
)

trip_analytical_dataset["gross_profit_uah"] = (
    trip_analytical_dataset["revenue_uah"] - trip_analytical_dataset["fuel_cost_uah"]
)

trip_analytical_dataset["gross_margin_pct"] = (
    trip_analytical_dataset["gross_profit_uah"] / trip_analytical_dataset["revenue_uah"] * 100
).round(1)

trip_analytical_dataset.head()

,trip_id,date_departure,date_arrival,status,client_id,client_name,client_region,route_id,origin_city,destination_city,...,driver_commission_rate,cargo_tons_actual,load_factor_pct,delay_hours,revenue_uah,estimated_fuel_liters,fuel_cost_uah,batches_used,gross_profit_uah,gross_margin_pct
0,TRP-00001,2022-07-01,2022-07-02,completed,CLT-02,ПП ДніпроЗерно,Дніпропетровська,RTE-02,Дніпро,Одеса,...,0.13,20.76,86.5,1.8,18715.5552,239.59,12985.78,FB-001,5729.7752,30.6
1,TRP-00002,2022-07-01,2022-07-01,completed,CLT-09,ФГ Золоте Колосся,Хмельницька,RTE-09,Хмельницький,Тернопіль,...,0.13,16.44,68.5,1.5,3829.8624,56.85,3081.27,FB-001,748.5924,19.5
2,TRP-00003,2022-07-01,2022-07-01,completed,CLT-09,ФГ Золоте Колосся,Хмельницька,RTE-09,Хмельницький,Тернопіль,...,0.13,16.74,68.3,1.1,3899.7504,54.93,2977.21,FB-001,922.5404,23.7
3,TRP-00004,2022-07-01,2022-07-01,delayed,CLT-08,ТОВ Пшеничний Шлях,Чернігівська,RTE-08,Чернігів,Київ,...,0.13,24.47,99.9,2.1,6526.3937,73.07,3960.39,FB-001,2566.0037,39.3
4,TRP-00005,2022-07-01,2022-07-01,completed,CLT-07,ФГ Жнива-Поділля,Вінницька,RTE-07,Вінниця,Жмеринка,...,0.13,19.47,79.5,0.4,1883.1384,25.50,1382.10,FB-001,501.0384,26.6


### Quick Quality Check

In [10]:
print(f"Rows: {len(trip_analytical_dataset):,}")
print(f"Columns: {trip_analytical_dataset.shape[1]}")

Rows: 4,925
Columns: 32


In [11]:
trip_analytical_dataset[
    ["revenue_uah", "fuel_cost_uah", "gross_profit_uah", "gross_margin_pct"]
].describe()

,revenue_uah,fuel_cost_uah,gross_profit_uah,gross_margin_pct
count,4880.000000,4827.000000,4827.000000,4827.000000
mean,12083.252492,5054.649540,7003.101215,57.833043
std,6735.052183,3292.820425,4020.803436,11.811952
min,1513.200000,628.990000,-8804.476000,-99.600000
25%,6303.832400,2448.885000,3585.279000,51.700000
50%,11689.725600,4255.680000,6873.106000,59.400000
75%,16507.775900,7016.870000,9766.762600,65.700000
max,29550.236400,28430.950000,21237.431600,83.100000


In [12]:
trip_analytical_dataset.loc[
    trip_analytical_dataset["gross_profit_uah"] < 0,
    ["trip_id", "trip_distance_km", "client_name", "route_type", "status", "cargo_tons_actual", "revenue_uah", "fuel_cost_uah", "gross_margin_pct"]
].sort_values("gross_margin_pct")

,trip_id,trip_distance_km,client_name,route_type,status,cargo_tons_actual,revenue_uah,fuel_cost_uah,gross_margin_pct
2593,TRP-02594,214.0,ТОВ Нива Експорт,highway,delayed,21.18,8838.4140,17642.89,-99.6
2604,TRP-02605,52.0,ФГ Жнива-Поділля,local,delayed,22.00,2276.5600,4286.94,-88.3
2606,TRP-02607,112.0,ФГ Золоте Колосся,local,delayed,25.73,5677.0672,9233.53,-62.6
2594,TRP-02595,52.0,ФГ Жнива-Поділля,local,delayed,27.05,2799.1340,4286.94,-53.2
2610,TRP-02611,112.0,ФГ Золоте Колосся,local,delayed,17.26,4252.8640,5831.48,-37.1
2630,TRP-02631,149.0,ТОВ Пшеничний Шлях,local,completed,18.47,6247.1081,7757.49,-24.2
3628,TRP-03629,612.0,ТОВ Полтавські Лани,highway,delayed,21.54,23201.1648,28283.35,-21.9
2609,TRP-02610,112.0,ФГ Золоте Колосся,local,delayed,21.37,4906.5520,5831.48,-18.9
2621,TRP-02622,112.0,ФГ Золоте Колосся,local,delayed,23.12,5101.1968,5831.48,-14.3
3618,TRP-03619,132.0,ФГ Кукурудза Південь,local,delayed,18.83,5393.6652,6100.26,-13.1


In [13]:
trip_analytical_dataset.groupby("status")["gross_margin_pct"].agg(["mean", "median", "count"])

,mean,median,count
status,,,
cancelled,NaN,NaN,0
completed,58.752365,60.1,4186
delayed,51.829485,54.3,641


### Finding — Delay Impact on Profitability

Delayed trips show consistently lower profitability than completed trips.

This is explained by idle fuel consumption during delays — trucks continue burning diesel while waiting, independent of distance travelled. 10 of the 11 trips with negative gross profit in this dataset are delayed trips, an extreme case of this same mechanism.

This finding will be revisited in `04_business_overview.ipynb` and explored in depth in the route delay analysis (Business Question 3).

In [14]:
print(f"Duplicate trip_id: {trip_analytical_dataset['trip_id'].duplicated().sum()}")
print(f"Negative gross_profit_uah: {(trip_analytical_dataset['gross_profit_uah'] < 0).sum()}")

Duplicate trip_id: 0
Negative gross_profit_uah: 11


In [15]:
print("Missing fuel cost:")
display(
    trip_analytical_dataset.loc[
        trip_analytical_dataset["fuel_cost_uah"].isna(),
        ["status"]
    ].value_counts()
)

Missing fuel cost:


status   
cancelled    45
completed    43
delayed      10
Name: count, dtype: int64

### QA Notes

- Fuel cost is missing only for cancelled trips and for trips occurring after a truck's last recorded refueling event, consistent with the missing `estimated_fuel_liters` pattern already confirmed in `02_trip_operational_dataset.ipynb`.
- No duplicate `trip_id` values and no negative gross profit rows are expected at this stage.

In [16]:
total_fifo_cost = trip_analytical_dataset["fuel_cost_uah"].sum()
total_purchased_cost = (fuel_batches["liters_purchased"] * fuel_batches["price_per_liter_uah"]).sum()
difference_pct = (total_purchased_cost - total_fifo_cost) / total_purchased_cost * 100

print(f"Total FIFO fuel cost (trips):        {total_fifo_cost:,.0f} UAH")
print(f"Total purchased fuel cost (batches): {total_purchased_cost:,.0f} UAH")
print(f"Difference:                          {total_purchased_cost - total_fifo_cost:,.0f} UAH ({difference_pct:.1f}%)")

Total FIFO fuel cost (trips):        24,398,793 UAH
Total purchased fuel cost (batches): 25,354,402 UAH
Difference:                          955,609 UAH (3.8%)


### Save Dataset

In [17]:
output_path = Path("../data/processed/trip_analytical_dataset.csv")

trip_analytical_dataset.to_csv(
    output_path,
    index=False
)

print(f"✅ Dataset saved to:\n{output_path}")

✅ Dataset saved to:
../data/processed/trip_analytical_dataset.csv


### Write Back to MySQL

`trip_analytical_dataset` is written back to MySQL as a table — the single source of truth Tableau connects to.

In [18]:
trip_analytical_dataset.to_sql(
    "trip_analytical_dataset",
    engine,
    if_exists="replace",
    index=False
)

print("✅ trip_analytical_dataset written to MySQL.")

✅ trip_analytical_dataset written to MySQL.


## Summary

The trip-level analytical dataset has been successfully created.

Output:

`data/processed/trip_analytical_dataset.csv`
MySQL table: `trip_analytical_dataset`

Key finding from this stage: delayed trips are consistently less profitable than completed trips (58.8% vs 51.8% mean gross margin), driven by idle fuel consumption during delays. This dataset becomes the single source of truth for `04_business_overview.ipynb`, all subsequent business-question notebooks, and the Tableau dashboard.